<a href="https://colab.research.google.com/github/VarshaP-0405/NLP-Project--Lecture2Revise/blob/main/Lecture2Revise_NLP_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**LECTURE2REVISE**

An NLP-Based Automatic Lecture-to-Revision Material Generation System

**Modules:**
1. Lecture Summarization
2. Key Point Extraction
3. Question Generation
4. Evaluation using ROUGE, BLEU and METEOR

In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [2]:
!pip install -q transformers datasets sentencepiece evaluate rouge_score nltk accelerate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


In [6]:
!pip install -q datasets

In [36]:
import nltk
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import sent_tokenize

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [4]:
!pip install -q datasets transformers sentencepiece evaluate rouge_score nltk

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.7 MB/s eta 0:00:00


In [5]:
import torch
from datasets import load_dataset

dataset = load_dataset("pszemraj/qmsum-cleaned")

print(dataset)
train_data = dataset["train"].shuffle(seed=42).select(
    range(min(1000, len(dataset["train"])))
)

val_data = dataset["validation"].shuffle(seed=42).select(
    range(min(150, len(dataset["validation"])))
)

test_data = dataset["test"].shuffle(seed=42).select(
    range(min(150, len(dataset["test"])))
)

print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))

README.md:   0%|          | 0.00/3.22k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 28.9MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.63MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1257 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/272 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/281 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'pid', 'input', 'output', 'input_token_count', 'output_token_count'],
        num_rows: 1257
    })
    validation: Dataset({
        features: ['id', 'pid', 'input', 'output', 'input_token_count', 'output_token_count'],
        num_rows: 272
    })
    test: Dataset({
        features: ['id', 'pid', 'input', 'output', 'input_token_count', 'output_token_count'],
        num_rows: 281
    })
})
Train: 1000
Validation: 150
Test: 150


# **Module 2 - Automatic Summarization**

In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model:", model_name)
print("Device:", device)
def preprocess(example):

    input_text = "summarize: " + example["input"]
    target_text = example["output"]

    model_input = tokenizer(
        input_text,
        max_length=512,
        truncation=True
    )

    labels = tokenizer(
        text_target=target_text,
        max_length=128,
        truncation=True
    )

    model_input["labels"] = labels["input_ids"]

    return model_input
tokenized_train = train_data.map(
    preprocess,
    batched=False
)

tokenized_val = val_data.map(
    preprocess,
    batched=False
)

print("Tokenization completed")
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./lecture2revise_t5",
    eval_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    num_train_epochs=3,
    save_strategy="epoch",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train()
trainer.save_model("./Lecture2Revise_Model")
tokenizer.save_pretrained("./Lecture2Revise_Model")

print("Summarization model saved")
sample = val_data[0]

inputs = tokenizer(
    "summarize: " + sample["input"],
    return_tensors="pt",
    max_length=512,
    truncation=True
).to(device)

with torch.no_grad():

    outputs = model.generate(
        **inputs,
        max_length=128,
        num_beams=4,
        no_repeat_ngram_size=3
    )

generated_summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("GENERATED SUMMARY:")
print(generated_summary)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Model: t5-small
Device: cuda
Tokenization completed


Epoch,Training Loss,Validation Loss
1,3.701068,3.484842
2,3.439063,3.430110
3,3.322035,3.420377


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Summarization model saved
GENERATED SUMMARY:
Industrial Designer thought that the remote control should be shaped like a spongy one. The design of the device should be different from the size of the other.


# **Module 3 - Key Point Extraction**

In [17]:
# ============================================================
# MODULE 3 - KEY POINT EXTRACTION
# ============================================================

import re
import nltk

from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download("punkt")
nltk.download("punkt_tab")


def clean_sentence(sentence):

    sentence = re.sub(
        r"\{[^}]*\}",
        "",
        sentence
    )

    sentence = re.sub(
        r"\s+",
        " ",
        sentence
    )

    return sentence.strip()


def extract_key_points(
    text,
    num_points=5
):

    sentences = sent_tokenize(text)

    cleaned = []

    for sentence in sentences:

        sentence = clean_sentence(sentence)

        if len(sentence.split()) >= 5:

            cleaned.append(sentence)


    # Remove duplicates

    unique_sentences = []

    for sentence in cleaned:

        if sentence.lower() not in [
            x.lower()
            for x in unique_sentences
        ]:

            unique_sentences.append(sentence)


    if not unique_sentences:

        return []


    # If fewer sentences than requested

    if len(unique_sentences) <= num_points:

        return unique_sentences


    # TF-IDF

    vectorizer = TfidfVectorizer(
        stop_words="english"
    )

    matrix = vectorizer.fit_transform(
        unique_sentences
    )

    scores = matrix.sum(axis=1).A1

    indices = scores.argsort()[::-1][:num_points]

    indices = sorted(indices)


    return [
        unique_sentences[i]
        for i in indices
    ]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# **Module 4 - Question Generation**

In [18]:
# ============================================================
# MODULE 4 - QUESTION GENERATION
# ============================================================

import re

def generate_question(key_point):

    point = key_point.strip()

    # Remove final punctuation
    point = re.sub(r"[.!?]+$", "", point)

    lower = point.lower()

    # --------------------------------------------------------
    # Pattern 1 - "X is a branch/subset/type of..."
    # --------------------------------------------------------

    if " is a branch of " in lower:

        subject = point.split(" is a branch of ")[0]

        return f"What is {subject}?"

    if " is a subset of " in lower:

        subject = point.split(" is a subset of ")[0]

        return f"What is {subject}?"


    # --------------------------------------------------------
    # Pattern 2 - "X focuses on..."
    # --------------------------------------------------------

    if "focuses on" in lower:

        subject = point.split(" focuses on ")[0]

        return f"What does {subject} focus on?"


    # --------------------------------------------------------
    # Pattern 3 - "X allows..."
    # --------------------------------------------------------

    if " allows " in lower:

        subject = point.split(" allows ")[0]

        return f"What does {subject} allow?"


    # --------------------------------------------------------
    # Pattern 4 - "X uses..."
    # --------------------------------------------------------

    if " uses " in lower:

        subject = point.split(" uses ")[0]

        return f"What does {subject} use?"


    # --------------------------------------------------------
    # Pattern 5 - "X is used for..."
    # --------------------------------------------------------

    if " is used for " in lower:

        subject = point.split(" is used for ")[0]

        return f"What is {subject} used for?"


    # --------------------------------------------------------
    # Pattern 6 - "X consists of..."
    # --------------------------------------------------------

    if " consists of " in lower:

        subject = point.split(" consists of ")[0]

        return f"What does {subject} consist of?"


    # --------------------------------------------------------
    # Default question
    # --------------------------------------------------------

    return f"What is the main idea of: {point}?"


# ------------------------------------------------------------
# Generate questions for all key points
# ------------------------------------------------------------

user_questions = []

for point in user_key_points:

    question = generate_question(point)

    user_questions.append(question)


# ------------------------------------------------------------
# Display questions
# ------------------------------------------------------------

print("=" * 70)
print("REVISION QUESTIONS")
print("=" * 70)

for i, question in enumerate(
    user_questions,
    1
):

    print(f"{i}. {question}")

REVISION QUESTIONS
1. What is Artificial intelligence?
2. What does It allow?


# **Module 5 - Revision Pack**

In [19]:
revision_pack = {
    "summary": generated_summary,
    "key_points": key_points,
    "questions": questions
}

print("=" * 60)
print("LECTURE2REVISE REVISION PACK")
print("=" * 60)

print("\nSUMMARY")
print("-" * 60)
print(generated_summary)

print("\nKEY POINTS")
print("-" * 60)

for i, point in enumerate(key_points, 1):
    print(f"{i}. {point}")

print("\nQUESTIONS")
print("-" * 60)

for i, question in enumerate(questions, 1):
    print(f"{i}. {question}")

LECTURE2REVISE REVISION PACK

SUMMARY
------------------------------------------------------------
Industrial Designer thought that the remote control should be shaped like a spongy one. The design of the device should be different from the size of the other.

KEY POINTS
------------------------------------------------------------
1. Industrial Designer thought that the remote control should be shaped a spongy one.
2. The design of the device should be different from the size of the other.

QUESTIONS
------------------------------------------------------------
1. Industrial Designer thought that the remote control should be shaped a spongy one?
2. The design of a device should be different from the size of the other?


# **Module 6 - Evaluvation**

In [20]:
import evaluate

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
predictions = []
references = []

for i in range(min(50, len(val_data))):

    sample = val_data[i]

    if not sample["output"].strip():
        continue

    inputs = tokenizer(
        "summarize: " + sample["input"],
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_length=128,
            num_beams=4,
            no_repeat_ngram_size=3
        )

    summary = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    predictions.append(summary)
    references.append(sample["output"])
rouge_results = rouge.compute(
    predictions=predictions,
    references=references
)

bleu_results = bleu.compute(
    predictions=predictions,
    references=[[r] for r in references]
)

meteor_results = meteor.compute(
    predictions=predictions,
    references=references
)

print("LECTURE2REVISE EVALUATION")
print("=" * 50)

print(
    "ROUGE-1:",
    round(rouge_results["rouge1"], 4)
)

print(
    "ROUGE-2:",
    round(rouge_results["rouge2"], 4)
)

print(
    "ROUGE-L:",
    round(rouge_results["rougeL"], 4)
)

print(
    "BLEU:",
    round(bleu_results["bleu"], 4)
)

print(
    "METEOR:",
    round(meteor_results["meteor"], 4)
)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


LECTURE2REVISE EVALUATION
ROUGE-1: 0.2889
ROUGE-2: 0.0969
ROUGE-L: 0.2034
BLEU: 0.0377
METEOR: 0.18


# **Module 7 - Enter Your Test**

In [23]:
# ============================================================
# MODULE 7 - USER LECTURE INPUT
# ============================================================

import re

print("=" * 70)
print("                 LECTURE2REVISE")
print("       Automatic Lecture Revision System")
print("=" * 70)

# ------------------------------------------------------------
# STEP 1 - USER INPUT
# ------------------------------------------------------------

lecture_text = input("\nPaste your lecture here:\n\n")

print("\nLecture received!")
print("Characters:", len(lecture_text))
print("Words:", len(lecture_text.split()))


# ------------------------------------------------------------
# STEP 2 - SUMMARIZATION
# ------------------------------------------------------------

inputs = tokenizer(
    "summarize: " + lecture_text,
    return_tensors="pt",
    max_length=512,
    truncation=True
).to(device)

with torch.no_grad():

    outputs = model.generate(
        **inputs,
        max_length=160,
        min_length=30,
        num_beams=5,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

final_summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)


# ------------------------------------------------------------
# STEP 3 - KEY POINT EXTRACTION
# ------------------------------------------------------------

user_key_points = extract_key_points(
    lecture_text,
    num_points=5
)


# ------------------------------------------------------------
# STEP 4 - QUESTION GENERATION
# ------------------------------------------------------------

def generate_question(point):

    point = point.strip()
    point = re.sub(r"[.!?]+$", "", point)

    lower = point.lower()

    if " is a branch of " in lower:

        subject = re.split(
            r"\s+is a branch of\s+",
            point,
            flags=re.IGNORECASE
        )[0]

        return f"What is {subject}?"

    if " is a subset of " in lower:

        subject = re.split(
            r"\s+is a subset of\s+",
            point,
            flags=re.IGNORECASE
        )[0]

        return f"What is {subject}?"

    if " allows " in lower:

        subject = re.split(
            r"\s+allows\s+",
            point,
            flags=re.IGNORECASE
        )[0]

        if subject.lower() == "it":
            return "What does machine learning allow?"

        return f"What does {subject} allow?"

    if " focuses on " in lower:

        subject = re.split(
            r"\s+focuses on\s+",
            point,
            flags=re.IGNORECASE
        )[0]

        return f"What does {subject} focus on?"

    if " consists of " in lower:

        subject = re.split(
            r"\s+consists of\s+",
            point,
            flags=re.IGNORECASE
        )[0]

        return f"What does {subject} consist of?"

    if " inspired by " in lower:

        subject = point.split(" are ")[0]

        return f"What are {subject} inspired by?"

    if " is widely used in " in lower:

        subject = re.split(
            r"\s+is widely used in\s+",
            point,
            flags=re.IGNORECASE
        )[0]

        return f"Where is {subject} widely used?"

    return f"What is the main idea of this statement?"


user_questions = []

for point in user_key_points:

    question = generate_question(point)

    user_questions.append(question)


# ------------------------------------------------------------
# STEP 5 - DISPLAY RESULTS
# ------------------------------------------------------------

print("\n")

print("=" * 70)
print("                         SUMMARY")
print("=" * 70)

print(final_summary)


print("\n")

print("=" * 70)
print("                  KEY REVISION POINTS")
print("=" * 70)

for i, point in enumerate(user_key_points, 1):

    print(f"{i}. {point}")


print("\n")

print("=" * 70)
print("                    REVISION QUESTIONS")
print("=" * 70)

for i, question in enumerate(user_questions, 1):

    print(f"{i}. {question}")


print("\n")

print("=" * 70)
print("              LECTURE2REVISE COMPLETE")
print("=" * 70)


                 LECTURE2REVISE
       Automatic Lecture Revision System

Paste your lecture here:

Artificial intelligence is a branch of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. Machine learning is a subset of artificial intelligence. It allows computers to learn patterns from data without being explicitly programmed. Deep learning is a further subset of machine learning that uses neural networks with multiple layers. Neural networks are computational models inspired by the human brain. They consist of interconnected nodes called neurons. Deep learning is widely used in image recognition, speech recognition, natural language processing, and autonomous vehicles.

Lecture received!
Characters: 650
Words: 91


                         SUMMARY
Artificial intelligence is a branch of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. It allows 